# Análisis Comparativo de Benchmarking GPU
## Comparación de las versiones CS_std, CS_1st y CS_2nd

Este notebook presenta un análisis visual comparativo de las tres versiones del sistema de rasterización.

In [ ]:
import json
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

# Configuración de estilo
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

In [ ]:
# Directorio de los archivos JSON
base_dir = r"d:\Users\Escritorio\Rasterization-Software\benchmark\nsight_analysis"

versiones = ['CS_std', 'CS_1st', 'CS_2nd']
moleculas = ['1aga', '1c0o', '2mjq', '8wql']
grids = ['grid1', 'grid2', 'grid3', 'grid4']

In [ ]:
def extraer_metricas_resumen(filepath):
    """Extrae métricas resumidas de un archivo JSON"""
    
    if not os.path.exists(filepath):
        return None
    
    with open(filepath, 'r') as f:
        data = json.load(f)
    
    tiempo_total = 0
    gr_cycles_list = []
    l1tex_list = []
    gpu_idle_list = []
    
    for range_key, range_data in data.items():
        for shader_data in range_data:
            tiempo_total += shader_data.get('Real_Duration_ms_Avg', 0)
            
            for rule in shader_data.get('Rules', []):
                for metric in rule.get('Metrics', []):
                    metric_id = metric.get('Id', '')
                    metric_value = metric.get('Value_Avg', 0)
                    
                    if metric_id == 'gr__cycles_active.avg.pct_of_peak_sustained_elapsed':
                        gr_cycles_list.append(metric_value)
                    elif metric_id == 'l1tex__t_sector_hit_rate.pct':
                        l1tex_list.append(metric_value)
                    elif metric_id == 'oracle.gr__cycles_idle.pct':
                        gpu_idle_list.append(metric_value)
    
    return {
        'tiempo_total': tiempo_total,
        'gr_cycles_avg': np.mean(gr_cycles_list) if gr_cycles_list else 0,
        'l1tex_avg': np.mean(l1tex_list) if l1tex_list else 0,
        'gpu_idle_avg': np.mean(gpu_idle_list) if gpu_idle_list else 0
    }

In [ ]:
# Recolectar datos de todas las versiones
data_moleculas = []
data_grids = []

# Moléculas
for mol in moleculas:
    for version in versiones:
        version_key = version.lower().replace('_', '_')
        filename = f"resultados_{version_key}_{mol}_ranges.json"
        filepath = os.path.join(base_dir, filename)
        
        metricas = extraer_metricas_resumen(filepath)
        if metricas:
            data_moleculas.append({
                'Molécula': mol,
                'Versión': version,
                'Tiempo Total (ms)': metricas['tiempo_total'],
                'GR Cycles Active [%]': metricas['gr_cycles_avg'],
                'L1TEX Hit Rate [%]': metricas['l1tex_avg'],
                'GPU Idle [%]': metricas['gpu_idle_avg']
            })

# Grids
for grid in grids:
    for version in versiones:
        version_key = version.lower().replace('_', '_')
        filename = f"resultados_{version_key}_{grid}_ranges.json"
        filepath = os.path.join(base_dir, filename)
        
        metricas = extraer_metricas_resumen(filepath)
        if metricas:
            data_grids.append({
                'Grid': grid,
                'Versión': version,
                'Tiempo Total (ms)': metricas['tiempo_total'],
                'GR Cycles Active [%]': metricas['gr_cycles_avg'],
                'L1TEX Hit Rate [%]': metricas['l1tex_avg'],
                'GPU Idle [%]': metricas['gpu_idle_avg']
            })

df_moleculas = pd.DataFrame(data_moleculas)
df_grids = pd.DataFrame(data_grids)

print("Datos de moléculas cargados:", len(df_moleculas))
print("Datos de grids cargados:", len(df_grids))

## 1. Comparación de Tiempos de Ejecución

In [ ]:
# Gráfico de barras: Tiempo total por molécula y versión
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Moléculas
pivot_moleculas = df_moleculas.pivot(index='Molécula', columns='Versión', values='Tiempo Total (ms)')
pivot_moleculas.plot(kind='bar', ax=ax1, width=0.8)
ax1.set_title('Tiempo de Ejecución por Molécula', fontsize=14, fontweight='bold')
ax1.set_ylabel('Tiempo Total (ms)', fontsize=12)
ax1.set_xlabel('Molécula', fontsize=12)
ax1.legend(title='Versión', fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.tick_params(axis='x', rotation=45)

# Grids
pivot_grids = df_grids.pivot(index='Grid', columns='Versión', values='Tiempo Total (ms)')
pivot_grids.plot(kind='bar', ax=ax2, width=0.8)
ax2.set_title('Tiempo de Ejecución por Grid', fontsize=14, fontweight='bold')
ax2.set_ylabel('Tiempo Total (ms)', fontsize=12)
ax2.set_xlabel('Grid', fontsize=12)
ax2.legend(title='Versión', fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 2. Métricas de GPU: GR Cycles Active

In [ ]:
# GR Cycles Active comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Moléculas
pivot_gr_mol = df_moleculas.pivot(index='Molécula', columns='Versión', values='GR Cycles Active [%]')
pivot_gr_mol.plot(kind='bar', ax=ax1, width=0.8, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
ax1.set_title('GR Cycles Active [%] por Molécula', fontsize=14, fontweight='bold')
ax1.set_ylabel('GR Cycles Active [%]', fontsize=12)
ax1.set_xlabel('Molécula', fontsize=12)
ax1.legend(title='Versión', fontsize=10)
ax1.axhline(y=80, color='r', linestyle='--', alpha=0.5, label='Target: 80%')
ax1.grid(True, alpha=0.3)
ax1.tick_params(axis='x', rotation=45)

# Grids
pivot_gr_grid = df_grids.pivot(index='Grid', columns='Versión', values='GR Cycles Active [%]')
pivot_gr_grid.plot(kind='bar', ax=ax2, width=0.8, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
ax2.set_title('GR Cycles Active [%] por Grid', fontsize=14, fontweight='bold')
ax2.set_ylabel('GR Cycles Active [%]', fontsize=12)
ax2.set_xlabel('Grid', fontsize=12)
ax2.legend(title='Versión', fontsize=10)
ax2.axhline(y=80, color='r', linestyle='--', alpha=0.5, label='Target: 80%')
ax2.grid(True, alpha=0.3)
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 3. L1TEX Hit Rate Comparison

In [ ]:
# L1TEX Hit Rate comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Moléculas
pivot_l1tex_mol = df_moleculas.pivot(index='Molécula', columns='Versión', values='L1TEX Hit Rate [%]')
pivot_l1tex_mol.plot(kind='bar', ax=ax1, width=0.8, color=['#d62728', '#9467bd', '#8c564b'])
ax1.set_title('L1TEX Hit Rate [%] por Molécula', fontsize=14, fontweight='bold')
ax1.set_ylabel('L1TEX Hit Rate [%]', fontsize=12)
ax1.set_xlabel('Molécula', fontsize=12)
ax1.legend(title='Versión', fontsize=10)
ax1.axhline(y=70, color='g', linestyle='--', alpha=0.5, label='Good: 70%')
ax1.grid(True, alpha=0.3)
ax1.tick_params(axis='x', rotation=45)

# Grids
pivot_l1tex_grid = df_grids.pivot(index='Grid', columns='Versión', values='L1TEX Hit Rate [%]')
pivot_l1tex_grid.plot(kind='bar', ax=ax2, width=0.8, color=['#d62728', '#9467bd', '#8c564b'])
ax2.set_title('L1TEX Hit Rate [%] por Grid', fontsize=14, fontweight='bold')
ax2.set_ylabel('L1TEX Hit Rate [%]', fontsize=12)
ax2.set_xlabel('Grid', fontsize=12)
ax2.legend(title='Versión', fontsize=10)
ax2.axhline(y=70, color='g', linestyle='--', alpha=0.5, label='Good: 70%')
ax2.grid(True, alpha=0.3)
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 4. GPU Idle Percentage

In [ ]:
# GPU Idle comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Moléculas
pivot_idle_mol = df_moleculas.pivot(index='Molécula', columns='Versión', values='GPU Idle [%]')
pivot_idle_mol.plot(kind='bar', ax=ax1, width=0.8, color=['#e377c2', '#7f7f7f', '#bcbd22'])
ax1.set_title('GPU Idle [%] por Molécula (Menor es Mejor)', fontsize=14, fontweight='bold')
ax1.set_ylabel('GPU Idle [%]', fontsize=12)
ax1.set_xlabel('Molécula', fontsize=12)
ax1.legend(title='Versión', fontsize=10)
ax1.axhline(y=10, color='g', linestyle='--', alpha=0.5, label='Target: <10%')
ax1.grid(True, alpha=0.3)
ax1.tick_params(axis='x', rotation=45)

# Grids
pivot_idle_grid = df_grids.pivot(index='Grid', columns='Versión', values='GPU Idle [%]')
pivot_idle_grid.plot(kind='bar', ax=ax2, width=0.8, color=['#e377c2', '#7f7f7f', '#bcbd22'])
ax2.set_title('GPU Idle [%] por Grid (Menor es Mejor)', fontsize=14, fontweight='bold')
ax2.set_ylabel('GPU Idle [%]', fontsize=12)
ax2.set_xlabel('Grid', fontsize=12)
ax2.legend(title='Versión', fontsize=10)
ax2.axhline(y=10, color='g', linestyle='--', alpha=0.5, label='Target: <10%')
ax2.grid(True, alpha=0.3)
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 5. Resumen Estadístico por Versión

In [ ]:
# Combinar datos de moléculas y grids
df_all = pd.concat([df_moleculas.rename(columns={'Molécula': 'Entidad'}), 
                    df_grids.rename(columns={'Grid': 'Entidad'})], ignore_index=True)

# Resumen por versión
resumen = df_all.groupby('Versión').agg({
    'Tiempo Total (ms)': ['mean', 'std', 'min', 'max'],
    'GR Cycles Active [%]': ['mean', 'std'],
    'L1TEX Hit Rate [%]': ['mean', 'std'],
    'GPU Idle [%]': ['mean', 'std']
}).round(2)

print("\n" + "="*80)
print("RESUMEN ESTADÍSTICO POR VERSIÓN")
print("="*80)
print(resumen)
print("\n")

## 6. Análisis de Correlaciones

In [ ]:
# Matriz de correlación para cada versión
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, version in enumerate(['CS_std', 'CS_1st', 'CS_2nd']):
    df_version = df_all[df_all['Versión'] == version]
    
    # Seleccionar solo columnas numéricas
    corr_cols = ['Tiempo Total (ms)', 'GR Cycles Active [%]', 'L1TEX Hit Rate [%]', 'GPU Idle [%]']
    corr = df_version[corr_cols].corr()
    
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                square=True, ax=axes[idx], cbar_kws={'shrink': 0.8})
    axes[idx].set_title(f'Correlaciones: {version}', fontsize=12, fontweight='bold')
    axes[idx].tick_params(axis='x', rotation=45)
    axes[idx].tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.show()

## 7. Comparación de Eficiencia: Throughput GPU

In [ ]:
# Calcular eficiencia: (100 - GPU Idle) / Tiempo Total
df_all['Eficiencia GPU'] = (100 - df_all['GPU Idle [%]']) / df_all['Tiempo Total (ms)']

# Gráfico de dispersión: Tiempo vs GPU Idle
fig, ax = plt.subplots(figsize=(12, 8))

for version in ['CS_std', 'CS_1st', 'CS_2nd']:
    df_v = df_all[df_all['Versión'] == version]
    ax.scatter(df_v['Tiempo Total (ms)'], df_v['GPU Idle [%]'], 
               s=100, alpha=0.6, label=version)

ax.set_xlabel('Tiempo Total (ms)', fontsize=12)
ax.set_ylabel('GPU Idle [%]', fontsize=12)
ax.set_title('Relación entre Tiempo de Ejecución y GPU Idle', fontsize=14, fontweight='bold')
ax.legend(title='Versión', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Conclusiones Finales

### Hallazgos Clave:

**CS_std (Versión Estándar):**
- ✅ **Tiempos más bajos** en la mayoría de casos
- ❌ Alto GPU Idle (~37%)
- ❌ Baja utilización de GR Cycles (~63%)
- ❌ L1TEX Hit Rate medio (~49%)

**CS_1st (Primera Optimización):**
- ✅ Excelente GR Cycles Active (~85%)
- ✅ GPU Idle reducido (~15%)
- ⚠️ Tiempos significativamente mayores
- ❌ L1TEX Hit Rate bajo (~43%)

**CS_2nd (Segunda Optimización):**
- ✅ Mejor GR Cycles Active (~90%)
- ✅ Excelente L1TEX Hit Rate (~72%)
- ✅ GPU Idle muy bajo (~10%)
- ❌ Tiempos variables, algunos muy altos

### Recomendaciones:

1. **Para producción rápida**: Usar **CS_std** si el tiempo absoluto es prioritario
2. **Para optimización GPU**: **CS_2nd** ofrece mejor uso de recursos GPU
3. **Balance**: **CS_1st** si se busca un compromiso
4. **Áreas de mejora**: 
   - CS_1st y CS_2nd tienen overhead de sincronización que aumenta tiempos
   - CS_std necesita mejor paralelización para reducir GPU Idle
   - CS_1st requiere optimización de acceso a caché (L1TEX)